# Tutorial for PUBMED queries + Claude 

Before we start with the code in this jupyter notebook, we first need to setup the Zotero API key and Anthropic API key in your enviroment

# Loading packages

First, we load the relevant packages

In [1]:
import argparse  # Parse command-line arguments to configure pipeline execution
import json  # Read and write structured data in JSON format
import os  # Interact with the operating system (files, paths, environment)
import xml.etree.ElementTree as ET  # Parse and navigate PubMed XML responses
from collections import defaultdict  # Dictionary with automatic default values
from datetime import datetime, timedelta  # Handle dates and date arithmetic
from time import sleep  # Pause execution to respect API rate limits
import anthropic  # Client library for interacting with Claude LLM API
import requests  # Send HTTP requests to PubMed and other REST APIs
from diskcache import Cache  # Persistent on-disk caching to avoid repeated API calls
from pyzotero import Zotero  # Interface with Zotero API for reference management

In [2]:
cache = Cache(".cache")  # Initialize persistent on-disk cache for API results and LLM outputs

zot = None  # Default to no Zotero connection unless credentials are provided
if "ZOTERO_API_KEY" in os.environ:
    zot = Zotero(
        "6377183", "group", os.environ["ZOTERO_API_KEY"]
    )  # Connect to Zotero group library using API key (cloud-based access)

# Initialize Claude client
api_key = os.environ.get("ANTHROPIC_API_KEY")  # Read Claude API key from environment variables

anthropic_client = None  # Default to no LLM client if API key is missing
if "ANTHROPIC_API_KEY" in os.environ:
    anthropic_client = anthropic.Anthropic(
        api_key=os.environ["ANTHROPIC_API_KEY"]
    )  # Create Claude client for paper classification and summarization

BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"  # Base URL for NCBI PubMed E-utilities API

# Search terms / Categories

In [ ]:
MAIN_QUERY = """(
  "machine learning"[MeSH Terms]
  OR "artificial intelligence"[MeSH Terms]
  OR "machine learning"[Title/Abstract]
  OR "artificial intelligence"[Title/Abstract]
  OR "deep learning"[Title/Abstract]
  OR "neural network*"[Title/Abstract]
  OR "random forest"[Title/Abstract]
  OR "support vector machine*"[Title/Abstract]
  OR "Gaussian process*"[Title/Abstract]
  OR "reinforcement learning"[Title/Abstract]
  OR "Bayesian machine learning"[Title/Abstract]
  OR "hybrid model*"[Title/Abstract]
  OR "mechanism-informed"[Title/Abstract]
)
AND
(
  pharmacokinetic*[Title/Abstract]
  OR pharmacodynamic*[Title/Abstract]
  OR PK/PD[Title/Abstract]
  OR "population pharmacokinetic*"[Title/Abstract]
  OR "nonlinear mixed effects"[Title/Abstract]
  OR NLME[Title/Abstract]
  OR PBPK[Title/Abstract]
  OR "physiologically based pharmacokinetic*"[Title/Abstract]
  OR QSP[Title/Abstract]
  OR "quantitative systems pharmacology"[Title/Abstract]
  OR "model-based drug development"[Title/Abstract]
  OR "model-informed drug development"[Title/Abstract]
  OR MIDD[Title/Abstract]
  OR pharmacometrics[Title/Abstract]
  OR "precision dosing"[Title/Abstract]
  OR "dose optimization"[Title/Abstract]
  OR "therapeutic drug monitoring"[Title/Abstract]
)
"""

FALLBACK_TAG = "Other/General"

# Updated mini-list of pharmacometrics applications
PMX_APPLICATION_TAGS = [
    "Outcome prediction",
    "Covariate selection / confounding adjustment",
    "Pharmacokinetic modeling",
    "Survival analysis",
    "Exposure–response analysis",
    "Pharmacodynamic modeling",
    "RWD phenotyping",
    "Drug toxicity prediction",
    "Drug repurposing",
    "Enrichment design",
    "Patient risk stratification / management",
    "Dose selection / optimization",
    "Adherence to drug regimen",
    "Synthetic control",
    "Postmarketing surveillance",
    "Endpoint / biomarker assessment",
    "Disease progression modeling",
    "Automation of PK/PD modeling",
    "Precision medicine / optimized treatment regimen",
    "Causal inference",
    "Data imputation",
    "Discovery of subpatient groups",
]

PAPER_TYPE_TAGS = ["review", "tutorial", "perspective"]

METHODOLOGY_TAGS = [
    "Supervised learning",
    "Unsupervised learning",
    "Deep learning",
    "Tree-based models",
    "Gaussian processes",
    "Bayesian ML",
    "Hybrid mechanistic–ML models",
    "Feature selection",
    "Model selection",
    "Surrogate modeling",
    "Emulation of NLME models",
    "Reinforcement learning",
    "Explainable AI",
    "Neural networks",
    "Ensemble learning",
    "Time-series modeling",
    "Mechanism-informed machine learning",
    "LLM",
    "AI Agents",
]

# Helper Functions

Let's go through each function one by one:

## ***get_pmids***: querying PubMed for recent papers

In [ ]:
def get_pmids(
    base_query: str,
    days_back: int = 1,
    max_results: int = 200,
):
    """
    Query PubMed using a fixed Boolean query plus a sliding publication date window.

    Parameters
    ----------
    base_query : str
        PubMed Boolean query (e.g. pharmacometrics OR clinical pharmacology)
    days_back : int
        How many days back from today to search
    max_results : int
        Maximum number of PMIDs to return

    Returns
    -------
    list[str]
        List of PubMed IDs (PMIDs)
    """

    # Get today's date in UTC (PubMed uses publication dates, not local time)
    end_date = datetime.now(UTC).date()

    # Compute the start date by subtracting days_back
    start_date = end_date - timedelta(days=days_back)

    # Construct PubMed date filter syntax
    # Example:
    # "2025/01/30"[Date - Publication] : "2025/01/31"[Date - Publication]
    date_clause = (
        f'"{start_date.strftime("%Y/%m/%d")}"[Date - Publication] : '
        f'"{end_date.strftime("%Y/%m/%d")}"[Date - Publication]'
    )

    # Combine the base query with the date constraint
    # Using parentheses ensures correct Boolean precedence
    full_query = f"""
    ({base_query})
    AND
    ({date_clause})
    """

    # PubMed E-utilities endpoint for searching
    search_url = f"{BASE_URL}esearch.fcgi"

    # Parameters passed to PubMed
    search_params = {
        "db": "pubmed",        # database to search
        "term": full_query,    # search query
        "retmax": max_results, # max number of results
        "retmode": "json",     # JSON response (easier to parse)
        "usehistory": "n",     # do not store query on NCBI servers
    }

    # Execute HTTP GET request with a timeout for safety
    response = requests.get(search_url, params=search_params, timeout=30)

    # Raise an exception if PubMed returns HTTP errors (4xx / 5xx)
    response.raise_for_status()

    # Parse JSON response into Python dict
    data = response.json()

    # Safely extract list of PMIDs
    # If any key is missing, return an empty list
    return data.get("esearchresult", {}).get("idlist", [])

In [ ]:
# Example: search for AI/ML pharmacometrics papers from the last 30 days
pmids = get_pmids(MAIN_QUERY, days_back=30, max_results=5)
print(f"Found {len(pmids)} PMIDs: {pmids}")

## ***get_article_entry***: helper to extract XML text safely

In [5]:
def get_article_entry(elem, key):
    """
    Extract text from an XML element using XPath-like syntax.

    Parameters
    ----------
    elem : xml.etree.ElementTree.Element
        Parent XML element (e.g. PubmedArticle)
    key : str
        Tag or XPath to search for

    Returns
    -------
    str or None
        Extracted text, or None if not found
    """

    # Find the first matching element
    res = elem.find(f".//{key}")

    # If found, return all text inside (handles nested tags)
    if res is not None:
        return "".join(res.itertext())

## ***query_pmid***: fetch and normalize one PubMed article

In [6]:
@cache.memoize()
def query_pmid(pmid):
    """
    Fetch full metadata for a single PubMed ID and convert it
    into a Zotero-compatible dictionary.

    Cached to avoid repeated API calls.
    """

    # PubMed efetch endpoint retrieves full records
    fetch_url = f"{BASE_URL}efetch.fcgi"

    # Parameters: database, PMID, XML output
    fetch_params = {"db": "pubmed", "id": pmid, "retmode": "xml"}

    # HTTP request to PubMed
    fetch_response = requests.get(fetch_url, params=fetch_params)

    # Parse XML response
    root = ET.fromstring(fetch_response.content)

    articles = []

    # Loop over PubmedArticle elements (normally exactly one per PMID)
    for article in root.findall(".//PubmedArticle"):

        # Extract PMID again from XML (safety)
        pmid = get_article_entry(article, "PMID")

        # Build a structured article dictionary
        article_dict = {
            "itemType": "journalArticle",  # Zotero item type
            "title": get_article_entry(article, "ArticleTitle"),
            "abstractNote": get_article_entry(article, "AbstractText"),
            "PMID": pmid,
            "date": get_article_entry(article, "PubDate"),
            "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
            "DOI": get_article_entry(article, "ArticleId[@IdType='doi']"),

            # Build list of authors
            "creators": [
                {
                    "creatorType": "author",
                    "firstName": author.findtext("ForeName"),
                    "lastName": author.findtext("LastName"),
                }
                for author in article.findall(".//AuthorList/Author")
            ],
        }

        articles.append(article_dict)

    # Sanity check: PubMed should not return multiple articles per PMID
    if len(articles) > 1:
        raise RuntimeError(f"Too many articles for given PMID ({pmid})")

    return articles[0]

In [ ]:
# Example: fetch metadata for the first PMID we found above
if pmids:
    article = query_pmid(pmids[0])
    print(f"Title:    {article['title']}")
    print(f"Date:     {article['date']}")
    print(f"DOI:      {article['DOI']}")
    print(f"Authors:  {', '.join(a['lastName'] + ' ' + (a['firstName'] or '') for a in article['creators'][:5])}")
    if article.get("abstractNote"):
        print(f"Abstract: {article['abstractNote'][:300]}...")
else:
    print("No PMIDs found in the previous step — try increasing days_back.")

## ***classify_paper***: LLM-based classification with Claude

In [38]:
# Optional caching
# @cache.memoize()  
def classify_paper(title, abstract):
    """
    Classify a paper into multiple axes: paper_type, application, methodology.
    Always returns an AI summary.

    Rules:
    - If all axes are empty, assign [Other/General]
    - If at least one axis has a tag, only use the tags returned by Claude
    - Summary is always returned
    """

    if abstract is None:
        abstract = ""

    # --- Claude not configured ---
    if anthropic_client is None:
        return (
            {
                "paper_type": [FALLBACK_TAG],
                "application": [FALLBACK_TAG],
                "methodology": [FALLBACK_TAG],
            },
            "Claude not configured"
        )

    # Build the prompt dynamically from external tag lists
    prompt = f"""
You are an experienced pharmacometrician with expertise in AI/ML applications in drug development and clinical pharmacology.

Classify the following paper into ZERO OR MORE tags per category.
Only assign a tag if clearly supported by the title or abstract.
Return STRICT JSON only.

Paper Title:
{title}

Abstract:
{abstract[:1500]}

Tag schema:

paper_type (choose any):
{chr(10).join(f"- {t}" for t in PAPER_TYPE_TAGS)}

application (choose any):
{chr(10).join(f"- {t}" for t in PMX_APPLICATION_TAGS)}

methodology (choose any):
{chr(10).join(f"- {t}" for t in METHODOLOGY_TAGS)}

Response format:
{{
  "paper_type": [],
  "application": [],
  "methodology": [],
  "summary": "one sentence summary (max 150 chars)"
}}
"""

    try:
        # --- Call Claude ---
        message = anthropic_client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=500,
            temperature=0,
            messages=[{"role": "user", "content": prompt}],
        )

        response_text = message.content[0].text

        # --- Default summary fallback ---
        summary = "AI/ML application in pharmacometrics or clinical pharmacology"

        # Try to parse JSON from Claude response
        try:
            start_idx = response_text.find("{")
            end_idx = response_text.rfind("}") + 1
            result = json.loads(response_text[start_idx:end_idx])
            summary = result.get("summary", summary)

        except Exception:
            result = {}

        # --- Extract tags per axis ---
        classification = {}
        for axis in ["paper_type", "application", "methodology"]:
            tags = result.get(axis, []) if isinstance(result, dict) else []
            classification[axis] = tags

        # --- If all axes empty, assign fallback Other/General only to application ---
        if all(len(tags) == 0 for tags in classification.values()):
            classification["application"] = [FALLBACK_TAG]

        return classification, summary

    except Exception as e:
        print(f"Error classifying paper '{title}': {e}")
        return (
            {
                "paper_type": [FALLBACK_TAG],
                "application": [FALLBACK_TAG],
                "methodology": [FALLBACK_TAG],
            },
            "Error during classification"
        )

In [ ]:
# Example: classify a paper using Claude (requires ANTHROPIC_API_KEY)
if pmids:
    article = query_pmid(pmids[0])
    classification, summary = classify_paper(article["title"], article.get("abstractNote", ""))
    print(f"Title:          {article['title']}")
    print(f"Summary:        {summary}")
    print(f"Paper type:     {classification['paper_type']}")
    print(f"Application:    {classification['application']}")
    print(f"Methodology:    {classification['methodology']}")
else:
    print("No PMIDs available — run the get_pmids example first.")

# Customizing PubMed Queries

PubMed's E-utilities accept Boolean queries with field tags that control where terms are matched. Understanding these building blocks lets you tune recall and precision for pharmacometrics literature searches.

## PubMed field tags

| Tag | Meaning | Example |
|-----|---------|---------|
| `[tiab]` or `[Title/Abstract]` | Match in title or abstract (free text) | `"deep learning"[tiab]` |
| `[mh]` or `[MeSH Terms]` | Match against MeSH controlled vocabulary | `"Machine Learning"[mh]` |
| `[au]` | Author name | `"Smith J"[au]` |
| `[dp]` | Date of publication | `"2024"[dp]` |
| `[pt]` | Publication type | `review[pt]` |
| `[ti]` | Title only | `"pharmacokinetics"[ti]` |

## Boolean operators

- **OR** — broaden: matches if *any* term is present
- **AND** — narrow: matches only if *all* terms are present
- **NOT** — exclude: removes matching results
- Use **parentheses** to group terms and control precedence

## Example: combining Boolean operators (AND / OR / NOT)

Build a query from two blocks joined with AND — an AI/ML block OR'd across method terms, and a pharmacometrics block OR'd across domain terms. Then exclude certain publication types with NOT.

In [ ]:
ML_BLOCK = """
(
  "machine learning"[tiab] OR
  "artificial intelligence"[tiab] OR
  "deep learning"[tiab] OR
  "neural network"[tiab] OR
  "random forest"[tiab] OR
  "gradient boosting"[tiab] OR
  "XGBoost"[tiab] OR
  "support vector"[tiab]
)
"""

PMX_BLOCK = """
(
  "pharmacometrics"[tiab] OR
  "population pharmacokinetic*"[tiab] OR
  "nonlinear mixed effect*"[tiab] OR
  "NONMEM"[tiab] OR
  "model-informed drug development"[tiab] OR
  pharmacokinetic*[tiab] OR
  pharmacodynamic*[tiab]
)
"""

EXCLUDE_BLOCK = """
NOT (
  review[pt] OR
  "electronic health record"[tiab] OR
  "EHR"[tiab] OR
  "radiomics"[tiab] OR
  "genome-wide"[tiab]
)
"""

CUSTOM_QUERY = f"""
{ML_BLOCK}
AND
{PMX_BLOCK}
{EXCLUDE_BLOCK}
"""

pmids_custom = get_pmids(CUSTOM_QUERY, days_back=365, max_results=20)
print(f"Custom AND/OR/NOT query: {len(pmids_custom)} papers found")

## Example: MeSH terms vs free-text search

MeSH (Medical Subject Headings) terms use a controlled vocabulary — PubMed indexes articles with MeSH descriptors, so `"Machine Learning"[mh]` catches papers even if the exact phrase doesn't appear in the title/abstract. Free-text `[tiab]` matches the literal string. MeSH is broader but requires the article to have been indexed.

In [ ]:
# MeSH-based query — uses controlled vocabulary
mesh_query = """
"Machine Learning"[mh] AND "Pharmacokinetics"[mh]
"""

# Free-text query — matches literal strings in title/abstract
freetext_query = """
"machine learning"[tiab] AND "pharmacokinetics"[tiab]
"""

pmids_mesh = get_pmids(mesh_query, days_back=365 * 5, max_results=200)
pmids_free = get_pmids(freetext_query, days_back=365 * 5, max_results=200)

print(f"MeSH query:      {len(pmids_mesh)} papers")
print(f"Free-text query: {len(pmids_free)} papers")
print(f"Overlap:         {len(set(pmids_mesh) & set(pmids_free))} papers in both")
print(f"MeSH-only:       {len(set(pmids_mesh) - set(pmids_free))} papers found only by MeSH")

## Example: date range filtering with `days_back`

The `get_pmids` function automatically adds a `[Date - Publication]` range filter. Adjusting `days_back` lets you control the search window.

In [ ]:
# Compare result counts across different time windows
simple_query = '"deep learning"[tiab] AND pharmacokinetic*[tiab]'

for window_days in [7, 90, 365, 365 * 3]:
    n = len(get_pmids(simple_query, days_back=window_days, max_results=500))
    label = f"{window_days} days" if window_days < 365 else f"{window_days // 365} year(s)"
    print(f"  Last {label:>10s}: {n} papers")

## Example: finding review papers

Use `review[pt]` (publication type) or free-text terms like `"systematic review"[tiab]` to target reviews, tutorials, and perspectives.

In [ ]:
REVIEW_QUERY = """
(
  "machine learning"[tiab] OR "artificial intelligence"[tiab] OR "deep learning"[tiab]
)
AND
(
  "pharmacometrics"[tiab] OR "model-based drug development"[tiab] OR
  "population pharmacokinetic*"[tiab] OR "MIDD"[tiab]
)
AND
(
  review[pt] OR
  "systematic review"[tiab] OR
  tutorial[tiab] OR
  perspective[tiab] OR
  "state of the art"[tiab] OR
  "practical guide"[tiab]
)
"""

pmids_reviews = get_pmids(REVIEW_QUERY, days_back=365 * 10, max_results=50)
print(f"Found {len(pmids_reviews)} review/tutorial/perspective papers\n")

for pmid in pmids_reviews[:5]:
    article = query_pmid(pmid)
    print(f"- {article['title']}")

## Example: full read-only pipeline (query, fetch, inspect)

Chain `get_pmids` → `query_pmid` → `classify_paper` for a small batch. No files are written and no external services are modified.

In [ ]:
# End-to-end read-only demo: search → fetch → classify
demo_query = '"reinforcement learning"[tiab] AND "dose optimization"[tiab]'
demo_pmids = get_pmids(demo_query, days_back=365 * 5, max_results=10)
print(f"Query returned {len(demo_pmids)} PMIDs\n")

for pmid in demo_pmids[:3]:
    article = query_pmid(pmid)
    classification, summary = classify_paper(
        article["title"], article.get("abstractNote", "")
    )
    print(f"Title:       {article['title']}")
    print(f"Date:        {article['date']}")
    print(f"Summary:     {summary}")
    print(f"Application: {classification['application']}")
    print(f"Methodology: {classification['methodology']}")
    print("-" * 80)